# Lab: OCR + RAG Chatbot (RapidOCR + Gemini)



### Step 1: Install Required Libraries

In [ ]:
# rapidocr        -> the OCR engine (runs fully on CPU via onnxruntime)
# onnxruntime      -> the backend RapidOCR uses to run its models
# google-genai     -> the official Gemini API SDK (chat + embeddings)
# requests, numpy  -> downloading files & doing the similarity math
!pip install rapidocr onnxruntime google-genai requests numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 29.0 MB/s eta 0:00:00


### Step 2: Import Libraries

In [ ]:
# Standard library
import os
import json

# For downloading the sample invoices
import requests
# for doing vector math
import numpy as np

# RapidOCR -- our OCR engine
from rapidocr import RapidOCR

# Gemini SDK -- used both for embeddings (retrieval) and chat (generation)
from google import genai

### Step 3: Set Up the Gemini API Key

In [ ]:
# Paste your key (get one for free at https://aistudio.google.com/apikey)
GEMINI_API_KEY = "YOUR GEMINI API KEY"

# Create one Gemini client we'll reuse for both embeddings and chat generation
client = genai.Client(api_key=GEMINI_API_KEY)

# Model names
CHAT_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "gemini-embedding-001"

print("Gemini client ready.")

Gemini client ready.


### Step 4: Download Sample Invoice Images from GitHub


In [ ]:
# Direct links to freely available sample documents on GitHub
INVOICE_URLS = {
    "simple-invoice.png": "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/simple-invoice.png",
    "contoso-receipt.png": "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/master/sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/receipt/contoso-receipt.png",
    "purchase-order-1.jpg": "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/master/sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/forms/Form_1.jpg",
}

os.makedirs("invoices", exist_ok=True)
invoice_paths = {}

for filename, url in INVOICE_URLS.items():
    path = os.path.join("invoices", filename)
    response = requests.get(url)
    response.raise_for_status()
    with open(path, "wb") as f:
        f.write(response.content)
    invoice_paths[filename] = path
    print(f"Downloaded {filename} ({len(response.content)/1024:.1f} KB)")

Downloaded simple-invoice.png (166.8 KB)
Downloaded contoso-receipt.png (1769.2 KB)
Downloaded purchase-order-1.jpg (468.0 KB)


### Step 5: Run OCR on Each Invoice (RapidOCR)


In [ ]:
# Initialize the OCR engine once -- this loads the detection, classification, and recognition models
engine = RapidOCR()

ocr_results = {}
for filename, path in invoice_paths.items():
    result = engine(path)
    ocr_results[filename] = result
    print(f"{filename}: found {len(result.txts)} text boxes in {result.elapse:.2f}s")

[INFO] 2026-07-28 07:33:17,245 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-28 07:33:17,360 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-28 07:33:17,363 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-28 07:33:17,501 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-28 07:33:17,509 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-28 07:33:17,511 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-28 07:33:17,616 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-28 07:33:17,741 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

simple-invoice.png: found 18 text boxes in 9.22s
contoso-receipt.png: found 19 text boxes in 7.38s
purchase-order-1.jpg: found 54 text boxes in 12.15s


### Step 6: Convert OCR Output into Structured Text

In [ ]:
# Build a dictionary of {filename: structured_text} -- this is our tiny document store
document_texts = {}
for filename, result in ocr_results.items():
    document_texts[filename] = result.to_markdown()

# Peek at one example to see the structure preserved
print(document_texts["simple-invoice.png"])

Contoso
Address:         Invoice For: Microsoft
1 Redmond way Suite    1020 Enterprise Way
6000 Redmond, WA     Sunnayvale, CA 87659
99243

Invoice Number Invoice Date Invoice Due Date Charges   VAT ID

34278587  6/18/2017  6/24/2017    $56,651.49 PT


### Step 7: Embed Each Document (Building the Knowledge Base)


In [ ]:
def embed_text(text):
    """Get a Gemini embedding vector for a piece of text."""
    response = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return np.array(response.embeddings[0].values)

# Build the knowledge base: one entry per document, with its text and embedding
knowledge_base = []
for filename, text in document_texts.items():
    knowledge_base.append({
        "source": filename,
        "text": text,
        "embedding": embed_text(text),
    })

print(f"Knowledge base built with {len(knowledge_base)} documents.")

Knowledge base built with 3 documents.


### Step 8: Define the Retrieval Function (Cosine Similarity)


In [ ]:
def cosine_similarity(a, b):
    # Standard cosine similarity: dot product over the product of magnitudes
    magnitude_a = np.sqrt(np.sum(a**2))
    magnitude_b = np.sqrt(np.sum(b**2))
    return np.dot(a, b) / (magnitude_a * magnitude_b)

def retrieve(query, top_k=2):
    """Return the top_k most relevant documents for the query, most relevant first."""
    query_embedding = embed_text(query)

    scored = []
    for doc in knowledge_base:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored.append({**doc, "score": float(score)})

    scored.sort(key=lambda d: d["score"], reverse=True)
    return scored[:top_k]

### Step 9: Build the RAG Pipeline (Retrieve + Ask Gemini)

In [ ]:
def rag_answer(query, top_k=2):
    retrieved_docs = retrieve(query, top_k=top_k) #top_k decides number of documents to fetch

    # Label each document so Gemini can cite exactly which one it used
    labeled_context = "\n\n".join(
        f"[Source: {doc['source']}]\n{doc['text']}" for doc in retrieved_docs
    )

    prompt = f"""You are an assistant that answers questions about invoices/receipts using ONLY the context below.
Every fact you state must be tagged with its source, like [Source: filename].
If the answer isn't in the context, say "Not found in the provided documents."

Context:
{labeled_context}

Question: {query}
Answer:"""

    response = client.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return response.text, retrieved_docs

### Step 10: Ask a Question

In [ ]:
query = "What is the invoice number and total charge on the Contoso invoice?"

answer, retrieved_docs = rag_answer(query)

print("--- DOCUMENTS RETRIEVED ---")
for doc in retrieved_docs:
    print(f"{doc['source']}  (similarity: {doc['score']:.3f})")

print("\n--- ANSWER ---")
print(answer)

--- DOCUMENTS RETRIEVED ---
simple-invoice.png  (similarity: 0.796)
contoso-receipt.png  (similarity: 0.772)

--- ANSWER ---
The invoice number on the Contoso invoice is 34278587 [Source: simple-invoice.png]. The total charge on the Contoso invoice is $56,651.49 [Source: simple-invoice.png].


### Step 11: Simple Interactive Chatbot

In [ ]:
while True:
    question = input("Ask about your invoices (or 'exit'): ").strip()
    if question.lower().strip() == "exit":
        print("Goodbye!")
        break

    answer, retrieved_docs = rag_answer(question)
    sources = ", ".join(doc["source"] for doc in retrieved_docs)
    print(f"\n[Retrieved: {sources}]")
    print(answer)
    print()

Ask about your invoices (or 'exit'): What is the contoso receipt about?

[Retrieved: contoso-receipt.png, simple-invoice.png]
The Contoso receipt is about a purchase made from Contoso [Source: contoso-receipt.png]. The items purchased include 1 Surface Pro 6 (256GB /Intel Core i5 / 8GB RAM (Black)) for $999.99 and 1 SurfacePen for $99.99 [Source: contoso-receipt.png]. The sub-total for these items was $1098.99, with an additional tax of $104.40, bringing the total to $1203.39 [Source: contoso-receipt.png]. The purchase was made on 6/10/2019 at 13:59, and the sales associate was Paul [Source: contoso-receipt.png]. Contoso's address is 123 Main Street, Redmond, WA 98052, and their phone number is 123-456-7890 [Source: contoso-receipt.png].

Ask about your invoices (or 'exit'): exit
Goodbye!


### Next Steps / Ideas

- Swap **RapidOCR** for **PaddleOCR** and compare accuracy/speed on the same images.
- Split longer documents into smaller overlapping chunks instead of one chunk per document.
- Ask Gemini to return **structured JSON** (invoice number, date, total, etc.) instead of free text, for downstream use.
- Add more invoices to `INVOICE_URLS` to see how retrieval scales with a bigger knowledge base.